In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 8.5 Final Model Selection and Deployment
- Recommended model: Survey-Enhanced XGBoost
- Deploy to hold-out students -> risk scores -> practical outreach threshold -> risk bands
- **Data note:** `Deploy_Survey_Data.csv` now includes a `SID` column — no separate names file needed for scoring.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import pickle

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load Deploy Data and Best Model
`Deploy_Survey_Data.csv` includes SID + the model's feature columns — split apart before scoring.

In [ ]:
df_deploy_full = pd.read_csv('../data/Deploy_Survey_Data.csv')
deploy_sids = df_deploy_full['SID']
df_deploy = df_deploy_full.drop(columns=['SID'])

survey_xgb_model = pickle.load(open('../models/Survey_xgb_model.pkl', 'rb'))
print("Model and deploy data loaded.")

## Generate Risk Scores

In [ ]:
holdout_prob = survey_xgb_model.predict_proba(df_deploy)[:, 1]
holdout_pred_default = (holdout_prob >= 0.50).astype(int)

holdout_scores = df_deploy.copy()
holdout_scores.insert(0, 'SID', deploy_sids.values)
holdout_scores['departure_risk_score'] = holdout_prob
holdout_scores['predicted_departed_default_050'] = holdout_pred_default

holdout_scores[['departure_risk_score', 'predicted_departed_default_050']].describe().round(4)

## Capacity-Based Outreach Threshold
A plain 0.50 cutoff may not match advising capacity. Flag the top 20% by risk instead — a practical, resource-aware threshold.

In [ ]:
capacity_share = 0.20
capacity_threshold = float(np.quantile(holdout_prob, 1 - capacity_share))
holdout_scores['flag_top_20pct_capacity'] = (holdout_scores['departure_risk_score'] >= capacity_threshold).astype(int)

print(f"Capacity threshold for top {capacity_share:.0%}: {capacity_threshold:.4f}")
print("Number flagged:", int(holdout_scores['flag_top_20pct_capacity'].sum()))

## Risk Bands for Stakeholders

In [ ]:
def assign_risk_band(score):
    if score >= np.quantile(holdout_prob, 0.80):
        return 'Priority outreach'
    elif score >= np.quantile(holdout_prob, 0.50):
        return 'Monitor/support'
    return 'Routine support'

holdout_scores['support_band'] = holdout_scores['departure_risk_score'].apply(assign_risk_band)
holdout_scores['support_band'].value_counts()

## Advisor-Facing Outreach List

In [ ]:
# Deploy_Survey_Data.csv now includes a real SID column, so holdout_scores
# already has each student's risk score correctly attached to their SID —
# no join against a separate names file is needed.
df_deploy_risk = holdout_scores

recommended_context_cols = ['SID', 'departure_risk_score', 'support_band',
    'flag_top_20pct_capacity', 'HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']

advisor_outreach_list = df_deploy_risk[recommended_context_cols].sort_values('departure_risk_score', ascending=False).reset_index(drop=True)
advisor_outreach_list.head(20).round(4)

## Summary
- Survey-Enhanced XGBoost deployed as a **decision-support tool**, not a decision-making system.
- Risk bands (not raw probabilities) make results actionable for advisors.
- The outreach list is keyed by SID, taken directly from `Deploy_Survey_Data.csv` — no join needed, no risk of misaligned rows.

**Module 8 Complete!**